# *Nonlinear Arterial Hemodynamics*
## Chapter 4 companion — Constitutive Anisotropy and Transverse Dynamics

This notebook is the computational companion to Chapter 4. It asks the same controlled question as the chapter:

**Can constitutive directionality alone open a transverse dynamical channel in the canonical straight, rigid Womersley problem?**

The book nomenclature governs all reader-facing text, equations, plots, tables, and exported files.

The notebook releases only the constitutive restriction. Curvature, torsion, branching, taper, axial development, wall motion, and the geometry-parameterized spectral dynamics of Chapter 5 remain withheld.

The canonical anisotropy coefficients used for verification are the book's Case A values. They are **verification parameters, not calibrated arterial population distributions**.

**Execution:** a clean Google Colab runtime should reproduce all outputs with **Run all** and no manual parameter choices.

### Chapter question

The classical Womersley state suppresses transverse kinematics by construction. Chapter 4 keeps the straight rigid tube but permits

$$
\mathbf u(r,t)
=
u_\theta(r,t)\mathbf e_\theta
+
u_z(r,t)\mathbf e_z.
$$

The notebook therefore has five responsibilities:

1. reproduce the coupled axial–azimuthal harmonic problem;
2. verify the Case A benchmark and the isotropic counterfactual;
3. expose the new axial-vorticity channel and the anisotropic Lamb-vector increment;
4. reconstruct nonlinear force-density spectra only after reconstructing the real fields;
5. use VascuQuest to place the **same fixed constitutive mechanism** across physiological Womersley conditions and waveform forcing.

VascuQuest does not supply or calibrate the anisotropy tensor.

### VascuQuest representation

VascuQuest/PWDB supplies, for each virtual subject and arterial site:

- age;
- heart rate;
- flow-velocity waveform;
- luminal-area waveform.

The native source signals are combined internally to recover the book quantity

$$
Q(t)=U(t)A(t),
$$

and the time-mean luminal area supplies the rigid reference radius

$$
R=\sqrt{\frac{\langle A\rangle_t}{\pi}}.
$$

The fundamental Womersley number is then

$$
\alpha=R\sqrt{\frac{\Omega}{\nu_{zz}}}.
$$

For population comparisons, the constitutive ratios are held fixed at the canonical Case A values

$$
\mathcal A_{z\theta}
=
\mathcal A_{\theta z}
=
0.1,
\qquad
\mathcal A_{\theta\theta}=1.
$$

This isolates the consequence of moving a single controlled constitutive mechanism through the physiological $R$ and $\Omega$ values represented in PWDB.

For the representative multiharmonic reconstruction, the observed $\widehat Q_m$ values determine the pressure-gradient amplitudes required by the coupled Chapter 4 model. The resulting $u_\theta$, $\omega_z$, $\Delta\ell_r$, and endothelial-scale descriptors are therefore **model-derived projections**, not native PWDB measurements.

In [ ]:
# Configuration and reproducibility constants
from pathlib import Path
import sys, json, subprocess, zipfile

ROOT = Path("/content/nonlinear_arterial_hemodynamics_ch04")
FIG_DIR = ROOT / "figures"
DATA_DIR = ROOT / "data"
META_DIR = ROOT / "metadata"
for d in (ROOT, FIG_DIR, DATA_DIR, META_DIR):
    d.mkdir(parents=True, exist_ok=True)

VQ_REPOSITORY = "https://github.com/KNOWDYN/VascuQuest.git"
VQ_GIT_REF = "8307147d72e7a6f3ea3135895bd6f52927c67439"
PWDB_RECORD_ID = "3275625"
PWDB_DOI = "10.5281/zenodo.3275625"

rho = 1060.0       # kg m^-3
mu = 3.5e-3        # Pa s

# Chapter 4 uses nu_zz as the reference kinematic viscosity.
nu_zz = mu / rho

# Canonical Case A constitutive ratios from Chapter 8.
A_ztheta = 0.1
A_thetaz = 0.1
A_thetatheta = 1.0

CASE_A_ALPHA = 8.0
CASE_A_M = 1
CASE_A_A_M = 1.0

SITES = [
    "AorticRoot", "ThorAorta", "AbdAorta",
    "Carotid", "Brachial", "Radial", "Femoral",
]

print("Working directory:", ROOT)
print(f"nu_zz = {nu_zz:.6e} m^2/s")

In [ ]:
# Install the pinned VascuQuest revision.
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    f"git+{VQ_REPOSITORY}@{VQ_GIT_REF}"
])

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from scipy.integrate import solve_bvp
from scipy.interpolate import PchipInterpolator
from scipy.special import jv
import vascuquest as vq

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("SciPy solve_bvp available")
print("matplotlib:", matplotlib.__version__)
print("VascuQuest:", getattr(vq, "__version__", "version field not exposed"))

In [ ]:
# Acquire and checksum-verify the PWDB artifacts used by this notebook.
ARTIFACTS = ["model_configurations", "common_site_waveforms_csv"]
verification = {}

for artifact in ARTIFACTS:
    subprocess.run(
        ["vascuquest", "dataset", "acquire",
         "--artifact", artifact, "--yes", "--format", "json"],
        check=True, text=True, capture_output=True,
    )
    verified = subprocess.run(
        ["vascuquest", "dataset", "verify",
         "--artifact", artifact, "--format", "json"],
        check=True, text=True, capture_output=True,
    )
    verification[artifact] = json.loads(verified.stdout)

status = subprocess.run(
    ["vascuquest", "dataset", "status", "--format", "json"],
    check=True, text=True, capture_output=True,
)
dataset_status = json.loads(status.stdout)
SOURCE_DIR = Path(dataset_status["managed_paths"]["source"])

(META_DIR / "artifact_verification.json").write_text(
    json.dumps(verification, indent=2), encoding="utf-8"
)
(META_DIR / "dataset_status.json").write_text(
    json.dumps(dataset_status, indent=2), encoding="utf-8"
)

print("Verified PWDB source:", SOURCE_DIR)

In [ ]:
# Open the verified dataset and establish deterministic subject metadata.
session = vq.open_dataset(source=SOURCE_DIR, offline=True)
assert session.identity.record_id == PWDB_RECORD_ID

age_result = session.get("age")
subject_ids = np.asarray(age_result.coordinates[0].values, dtype=str)
ages = np.asarray(age_result.values, dtype=float)

hr_result = session.get("heart_rate", subjects=subject_ids.tolist())
hr_ids = np.asarray(hr_result.coordinates[0].values, dtype=str)
heart_rates = np.asarray(hr_result.values, dtype=float)
assert np.array_equal(subject_ids, hr_ids)

subject_meta = pd.DataFrame({
    "subject_id": subject_ids,
    "age_years": ages,
    "heart_rate_bpm": heart_rates,
})
subject_meta["subject_number"] = subject_meta["subject_id"].astype(int)
subject_meta = subject_meta.sort_values("subject_number").reset_index(drop=True)

# Deterministic representative subject:
# middle source age stratum, then median canonical subject number.
source_ages = sorted(subject_meta["age_years"].dropna().unique())
target_age = source_ages[len(source_ages)//2]
age_group = subject_meta.loc[subject_meta["age_years"] == target_age].copy()
age_group = age_group.sort_values("subject_number").reset_index(drop=True)
representative_subject = str(age_group.iloc[len(age_group)//2]["subject_id"])

selection_record = {
    "rule": "middle PWDB source age stratum; median canonical subject number",
    "representative_subject_id": representative_subject,
    "representative_age_years": float(target_age),
    "source_age_strata_years": [float(x) for x in source_ages],
}
(META_DIR / "representative_subject.json").write_text(
    json.dumps(selection_record, indent=2), encoding="utf-8"
)

display(pd.DataFrame([selection_record]))

In [ ]:
# Shared B&W plotting system and VascuQuest waveform reader.
WAVE_ZIP = SOURCE_DIR / "PWs_csv.zip"

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["DejaVu Serif"],
    "mathtext.fontset": "stix",
    "font.size": 9.0,
    "axes.labelsize": 9.0,
    "axes.titlesize": 9.5,
    "xtick.labelsize": 8.0,
    "ytick.labelsize": 8.0,
    "legend.fontsize": 7.8,
    "axes.linewidth": 0.75,
    "lines.linewidth": 1.2,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})
BLACK, DARK, MID, LIGHT = "0.0", "0.28", "0.52", "0.74"

SITE_LABELS = {
    "AorticRoot": "Aortic root",
    "ThorAorta": "Thoracic aorta",
    "AbdAorta": "Abdominal aorta",
    "Carotid": "Carotid",
    "Brachial": "Brachial",
    "Radial": "Radial",
    "Femoral": "Femoral",
}

def clean_axes(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(False)

def save_figure(fig, stem):
    pdf = FIG_DIR / f"{stem}.pdf"
    png = FIG_DIR / f"{stem}.png"
    fig.savefig(pdf, bbox_inches="tight", pad_inches=0.03)
    fig.savefig(png, dpi=600, bbox_inches="tight", pad_inches=0.03)
    return pdf, png

def _wave_member_name(site_id, source_signal):
    basename = f"PWs_{site_id}_{source_signal}.csv"
    with zipfile.ZipFile(WAVE_ZIP, "r") as zf:
        matches = [name for name in zf.namelist() if Path(name).name == basename]
    if len(matches) != 1:
        raise RuntimeError(f"Expected one {basename!r}; found {len(matches)}")
    return matches[0]

def load_waveform_matrix(site_id, source_signal):
    # Database-native field names are confined to this mapping layer.
    member = _wave_member_name(site_id, source_signal)
    with zipfile.ZipFile(WAVE_ZIP, "r") as zf:
        with zf.open(member, "r") as raw:
            frame = pd.read_csv(raw, low_memory=False)
    ids = np.asarray([str(int(x)) for x in frame.iloc[:, 0].to_numpy()], dtype=str)
    values = frame.iloc[:, 1:].to_numpy(dtype=float)
    return ids, values

print("Shared helpers ready.")

# The reference kinematics and what is being changed

Retain

$$
u_r=0,
\qquad
\frac{\partial}{\partial\theta}=0,
\qquad
\frac{\partial}{\partial z}=0,
$$

but no longer impose $u_\theta=0$.

The admissible velocity is

$$
\mathbf u(r,t)
=
u_\theta(r,t)\mathbf e_\theta
+
u_z(r,t)\mathbf e_z.
$$

Curvature, branching, torsion, taper, axial development, and wall motion remain fixed at their reference values. Within this controlled problem, any nonzero $u_\theta$ must therefore arise from the constitutive coupling.

# From scalar viscosity to a coupled shear law

Define

$$
S_z=\frac{\partial u_z}{\partial r},
\qquad
S_\theta=
\frac{\partial u_\theta}{\partial r}
-
\frac{u_\theta}{r}.
$$

The reduced anisotropic constitutive law is

$$
\begin{bmatrix}
\tau_{zr}\\[1mm]
\tau_{\theta r}
\end{bmatrix}
=
\rho
\begin{bmatrix}
\nu_{zz} & \nu_{z\theta}\\
\nu_{\theta z} & \nu_{\theta\theta}
\end{bmatrix}
\begin{bmatrix}
S_z\\[1mm]
S_\theta
\end{bmatrix}.
$$

The dimensionless ratios are

$$
\mathcal A_{z\theta}
=
\frac{\nu_{z\theta}}{\nu_{zz}},
\qquad
\mathcal A_{\theta z}
=
\frac{\nu_{\theta z}}{\nu_{zz}},
\qquad
\mathcal A_{\theta\theta}
=
\frac{\nu_{\theta\theta}}{\nu_{zz}}.
$$

Dissipative admissibility requires the symmetric part of the viscosity matrix to be positive definite. For the symmetric canonical sweep used below,

$$
\mathcal A_{z\theta}
=
\mathcal A_{\theta z}
=
\mathcal A_c,
\qquad
\mathcal A_{\theta\theta}=1,
$$

so admissibility reduces to

$$
|\mathcal A_c|<1.
$$

This sweep is a controlled model sensitivity study. It is not a claim about measured arterial anisotropy.

# Coupled axial and azimuthal momentum

The chapter defines four distinct cylindrical operators,

$$
\mathcal D_0 f
=
\frac{d^2f}{dr^2}
+
\frac1r\frac{df}{dr},
$$

$$
\mathcal D_1 f
=
\frac{d^2f}{dr^2}
+
\frac1r\frac{df}{dr}
-
\frac{f}{r^2},
$$

$$
\mathcal D_{z\theta}f
=
\frac{d^2f}{dr^2},
$$

$$
\mathcal D_{\theta z}f
=
\frac{d^2f}{dr^2}
+
\frac2r\frac{df}{dr}.
$$

For harmonic $m$,

$$
i m\alpha^2\widehat U_{z,m}
=
a_m
+
\mathcal D_0^*\widehat U_{z,m}
+
\mathcal A_{z\theta}
\mathcal D_{z\theta}^*
\widehat U_{\theta,m},
$$

$$
i m\alpha^2\widehat U_{\theta,m}
=
\mathcal A_{\theta z}
\mathcal D_{\theta z}^*
\widehat U_{z,m}
+
\mathcal A_{\theta\theta}
\mathcal D_1^*
\widehat U_{\theta,m}.
$$

The pressure forcing appears only in the axial equation. The azimuthal response is generated indirectly through the off-diagonal constitutive coefficients.

Chapter 8 describes Chebyshev collocation as the production discretization and also reports an independent adaptive boundary-value verification. This notebook uses that adaptive boundary-value route because it provides a compact, transparent implementation of the Chapter 4 equations while preserving an independent check against the published Case A benchmark.

In [ ]:
# Adaptive boundary-value solver for the dimensionless Chapter 4 system.
#
# The apparent cylindrical singularities are handled by solving on [epsilon, 1]
# and imposing the analytic regularity branch at epsilon:
# dU_z/dx -> 0 and U_theta -> 0 as x -> 0.
#
# The epsilon sensitivity is checked explicitly against the Chapter 8 benchmark.
def solve_ch4_harmonic(
    alpha,
    m=1,
    a_m=1.0,
    Azt=A_ztheta,
    Atz=A_thetaz,
    Att=A_thetatheta,
    epsilon=1e-5,
    tol=1e-7,
    initial_nodes=320,
    max_nodes=20000,
):
    lam = m * alpha**2
    coupling = np.array([[1.0, Azt], [Atz, Att]], dtype=float)
    if abs(np.linalg.det(coupling)) < 1e-10:
        raise ValueError("Constitutive second-derivative block is singular.")
    inv_coupling = np.linalg.inv(coupling)

    x = np.linspace(epsilon, 1.0, initial_nodes)

    def ode(x, y):
        Uz = y[0] + 1j*y[4]
        Uzp = y[1] + 1j*y[5]
        Uth = y[2] + 1j*y[6]
        Uthp = y[3] + 1j*y[7]

        # Rearranged forms of Eqs. ch4_nd_z and ch4_nd_theta.
        rhs_z = 1j*lam*Uz - a_m - Uzp/x
        rhs_th = (
            1j*lam*Uth
            - 2.0*Atz*Uzp/x
            - Att*Uthp/x
            + Att*Uth/x**2
        )

        second = inv_coupling @ np.vstack([rhs_z, rhs_th])
        Uzpp, Uthpp = second[0], second[1]

        dy = np.empty_like(y)
        vals = [Uzp, Uzpp, Uthp, Uthpp]
        for k, value in enumerate(vals):
            dy[k] = value.real
            dy[k+4] = value.imag
        return dy

    def bc(ya, yb):
        Uzp_a = ya[1] + 1j*ya[5]
        Uth_a = ya[2] + 1j*ya[6]
        Uz_b = yb[0] + 1j*yb[4]
        Uth_b = yb[2] + 1j*yb[6]

        residuals = [Uzp_a, Uth_a, Uz_b, Uth_b]
        return np.array(
            [z.real for z in residuals] +
            [z.imag for z in residuals]
        )

    # Smooth initial guess consistent with the boundary conditions.
    y0 = np.zeros((8, x.size))
    y0[0] = (1.0 - x**2) / (1.0 + alpha**2)
    y0[1] = -2.0*x / (1.0 + alpha**2)

    sol = solve_bvp(
        ode, bc, x, y0,
        tol=tol, max_nodes=max_nodes, verbose=0
    )
    if sol.status != 0:
        raise RuntimeError(sol.message)

    return sol

def evaluate_solution(sol, x):
    y = sol.sol(np.asarray(x))
    Uz = y[0] + 1j*y[4]
    Uzp = y[1] + 1j*y[5]
    Uth = y[2] + 1j*y[6]
    Uthp = y[3] + 1j*y[7]
    return Uz, Uzp, Uth, Uthp

print("Chapter 4 adaptive BVP solver ready.")

In [ ]:
# Reproduce the canonical Case A benchmark from Chapter 8.
eps_values = [1e-3, 1e-4, 1e-5]
benchmark_rows = []

for eps in eps_values:
    sol = solve_ch4_harmonic(
        alpha=CASE_A_ALPHA,
        m=CASE_A_M,
        a_m=CASE_A_A_M,
        epsilon=eps,
        tol=1e-7,
    )
    x_eval = np.linspace(eps, 1.0, 1600)
    Uz, Uzp, Uth, Uthp = evaluate_solution(sol, x_eval)

    benchmark_rows.append({
        "epsilon": eps,
        "adaptive_nodes": len(sol.x),
        "max_abs_Uz": float(np.max(np.abs(Uz))),
        "max_abs_Utheta": float(np.max(np.abs(Uth))),
    })

benchmark_df = pd.DataFrame(benchmark_rows)
benchmark_df.to_csv(DATA_DIR / "ch04_caseA_epsilon_check.csv", index=False)
display(benchmark_df)

# Chapter 8 reference values at tight tolerance.
book_max_Uz = 0.017052742
book_max_Utheta = 4.973726e-4

final_row = benchmark_df.iloc[-1]
print("Relative difference from Chapter 8 benchmark:")
print(
    "max|U_z|:",
    abs(final_row["max_abs_Uz"] - book_max_Uz) / book_max_Uz
)
print(
    "max|U_theta|:",
    abs(final_row["max_abs_Utheta"] - book_max_Utheta) / book_max_Utheta
)

The Chapter 8 Case A benchmark is

$$
\alpha=8,
\qquad
m=1,
\qquad
a_m=1,
$$

$$
\mathcal A_{z\theta}
=
\mathcal A_{\theta z}
=
0.1,
\qquad
\mathcal A_{\theta\theta}=1.
$$

The reported converged amplitudes are

$$
\max|\widehat U_z|
=
0.017052742,
$$

$$
\max|\widehat U_\theta|
=
4.973726\times10^{-4}.
$$

The previous cell checks the notebook implementation against those values while also checking sensitivity to the small numerical offset used to represent the analytic centreline regularity branch.

In [ ]:
# Plot the canonical Case A velocity components.
sol_caseA = solve_ch4_harmonic(
    alpha=CASE_A_ALPHA,
    m=1,
    a_m=1.0,
    epsilon=1e-5,
    tol=1e-8,
)
x = np.linspace(1e-5, 1.0, 1000)
Uz, Uzp, Uth, Uthp = evaluate_solution(sol_caseA, x)

fig, axes = plt.subplots(1, 2, figsize=(7.1, 3.1))

axes[0].plot(x, np.abs(Uz)/np.max(np.abs(Uz)), color=BLACK, linestyle="-")
axes[0].set_xlabel(r"Normalized radius, $x=r/R$")
axes[0].set_ylabel(r"$|\widehat U_{z,1}|/\max|\widehat U_{z,1}|$")
clean_axes(axes[0])

axes[1].plot(x, np.abs(Uth)/np.max(np.abs(Uth)), color=BLACK, linestyle="-")
axes[1].set_xlabel(r"Normalized radius, $x=r/R$")
axes[1].set_ylabel(r"$|\widehat U_{\theta,1}|/\max|\widehat U_{\theta,1}|$")
clean_axes(axes[1])

fig.tight_layout(w_pad=1.4)
save_figure(fig, "ch04_caseA_velocity_components")
plt.show()

This reproduces the mechanism figure already present in Chapter 4. It is primarily a verification figure for the notebook and should replace, not duplicate, the existing book figure if the notebook-generated rendering is preferred.

# The counterfactual: switch the anisotropy off

Set

$$
\mathcal A_{z\theta}
=
\mathcal A_{\theta z}
=
0,
\qquad
\mathcal A_{\theta\theta}=1.
$$

Then

$$
\widehat U_{\theta,m}=0,
$$

and the axial equation collapses to the scalar-viscosity Womersley problem.

The strongest computational test is therefore a mechanism-off calculation: the transverse component must vanish and the axial component must recover the exact Bessel solution.

In [ ]:
# Isotropic recovery against the exact Chapter 3 Womersley solution.
def Lambda_W(alpha_m):
    return np.exp(3j*np.pi/4.0) * alpha_m

def isotropic_dimensionless_Uz(x, alpha, m=1, a_m=1.0):
    # From the Chapter 3 Bessel solution after Chapter 4 nondimensionalization.
    alpha_m = np.sqrt(m) * alpha
    LW = Lambda_W(alpha_m)
    return (a_m/(1j*m*alpha**2)) * (
        1.0 - jv(0, LW*x)/jv(0, LW)
    )

sol_iso = solve_ch4_harmonic(
    alpha=CASE_A_ALPHA,
    m=1,
    a_m=1.0,
    Azt=0.0,
    Atz=0.0,
    Att=1.0,
    epsilon=1e-5,
    tol=1e-9,
)
x_iso = np.linspace(1e-5, 1.0, 1500)
Uz_iso_num, _, Uth_iso_num, _ = evaluate_solution(sol_iso, x_iso)
Uz_iso_exact = isotropic_dimensionless_Uz(x_iso, CASE_A_ALPHA)

relative_Uz_error = (
    np.max(np.abs(Uz_iso_num - Uz_iso_exact))
    / np.max(np.abs(Uz_iso_exact))
)
max_Utheta_iso = np.max(np.abs(Uth_iso_num))

print("Relative axial recovery error:", relative_Uz_error)
print("max|U_theta| in isotropic limit:", max_Utheta_iso)

fig, axes = plt.subplots(1, 2, figsize=(7.1, 3.05))

axes[0].plot(
    x_iso, np.abs(Uz_iso_exact),
    color=LIGHT, linestyle=":", linewidth=2.0,
    label="exact isotropic Womersley"
)
axes[0].plot(
    x_iso, np.abs(Uz_iso_num),
    color=BLACK, linestyle="-",
    label="coupled solver, anisotropy off"
)
axes[0].set_xlabel(r"Normalized radius, $x=r/R$")
axes[0].set_ylabel(r"$|\widehat U_{z,1}|$")
axes[0].legend(frameon=False)
clean_axes(axes[0])

axes[1].semilogy(
    x_iso, np.maximum(np.abs(Uth_iso_num), 1e-18),
    color=BLACK
)
axes[1].set_xlabel(r"Normalized radius, $x=r/R$")
axes[1].set_ylabel(r"$|\widehat U_{\theta,1}|$")
clean_axes(axes[1])

fig.tight_layout(w_pad=1.4)
save_figure(fig, "ch04_isotropic_recovery")
plt.show()

The mechanism-off result is more important than a generic convergence plot because it tests the physical architecture of the model:

- the azimuthal response disappears;
- the axial response returns to the exact scalar-viscosity Womersley solution;
- no geometry change is involved.

The transverse channel is therefore attributable to the released constitutive degree of freedom in this controlled problem.

# Sensitivity to the constitutive coupling

The off-diagonal coefficients are the new ingredient. To isolate their effect without inventing a physiological distribution, sweep the symmetric coupling

$$
\mathcal A_{z\theta}
=
\mathcal A_{\theta z}
=
\mathcal A_c
$$

through a controlled admissible interval while holding

$$
\mathcal A_{\theta\theta}=1.
$$

For this symmetric choice, positive definiteness requires

$$
|\mathcal A_c|<1.
$$

The sweep below is therefore a model sensitivity study, not a population inference.

In [ ]:
# Canonical coupling sweep at fixed alpha=8.
couplings = np.linspace(0.0, 0.45, 19)
sweep_rows = []

for Ac in couplings:
    sol = solve_ch4_harmonic(
        alpha=CASE_A_ALPHA,
        m=1,
        a_m=1.0,
        Azt=Ac,
        Atz=Ac,
        Att=1.0,
        epsilon=1e-5,
        tol=3e-7,
    )
    xx = np.linspace(1e-5, 1.0, 700)
    Uz_c, _, Uth_c, _ = evaluate_solution(sol, xx)

    sweep_rows.append({
        "A_c": Ac,
        "max_abs_Uz": np.max(np.abs(Uz_c)),
        "max_abs_Utheta": np.max(np.abs(Uth_c)),
        "transverse_to_axial_ratio":
            np.max(np.abs(Uth_c))/np.max(np.abs(Uz_c)),
    })

coupling_df = pd.DataFrame(sweep_rows)
coupling_df.to_csv(DATA_DIR / "ch04_coupling_sweep.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(7.1, 3.05))

axes[0].plot(
    coupling_df["A_c"],
    coupling_df["transverse_to_axial_ratio"],
    color=BLACK, marker="o", markersize=3.5
)
axes[0].set_xlabel(r"Symmetric coupling, $\mathcal A_c$")
axes[0].set_ylabel(r"$\max|\widehat U_\theta|/\max|\widehat U_z|$")
clean_axes(axes[0])

axes[1].plot(
    coupling_df["A_c"],
    coupling_df["max_abs_Uz"],
    color=BLACK, linestyle="-",
    label=r"$\max|\widehat U_z|$"
)
axes[1].plot(
    coupling_df["A_c"],
    coupling_df["max_abs_Utheta"],
    color=DARK, linestyle="--",
    label=r"$\max|\widehat U_\theta|$"
)
axes[1].set_xlabel(r"Symmetric coupling, $\mathcal A_c$")
axes[1].set_ylabel("Dimensionless harmonic amplitude")
axes[1].legend(frameon=False)
clean_axes(axes[1])

fig.tight_layout(w_pad=1.3)
save_figure(fig, "ch04_coupling_sensitivity")
plt.show()

# Vorticity opened by the azimuthal mode

For the Chapter 4 kinematics,

$$
\boldsymbol\omega
=
\omega_\theta\mathbf e_\theta
+
\omega_z\mathbf e_z,
$$

with

$$
\omega_\theta
=
-\frac{\partial u_z}{\partial r},
$$

$$
\omega_z
=
\frac1r
\frac{\partial}{\partial r}
(r u_\theta).
$$

The classical Womersley state already contains $\omega_\theta$. The constitutive extension creates the additional axial-vorticity channel $\omega_z$ by making $u_\theta$ admissible.

In [ ]:
# Canonical harmonic vorticity fields.
# For dimensionless variables, multiplying by U0/R would restore physical units.
omega_theta_hat = -Uzp
omega_z_hat = Uthp + Uth/x

fig, axes = plt.subplots(1, 2, figsize=(7.1, 3.05))

axes[0].plot(
    x, np.abs(omega_theta_hat)/np.max(np.abs(omega_theta_hat)),
    color=BLACK
)
axes[0].set_xlabel(r"Normalized radius, $x=r/R$")
axes[0].set_ylabel(r"$|\widehat\omega_{\theta,1}|/\max|\widehat\omega_{\theta,1}|$")
clean_axes(axes[0])

axes[1].plot(
    x, np.abs(omega_z_hat)/np.max(np.abs(omega_z_hat)),
    color=BLACK
)
axes[1].set_xlabel(r"Normalized radius, $x=r/R$")
axes[1].set_ylabel(r"$|\widehat\omega_{z,1}|/\max|\widehat\omega_{z,1}|$")
clean_axes(axes[1])

fig.tight_layout(w_pad=1.4)
save_figure(fig, "ch04_vorticity_channels")
plt.show()

# Lamb-vector baseline and anisotropic increment

The Lamb vector is

$$
\boldsymbol\ell
=
\mathbf u\times\boldsymbol\omega.
$$

For the present kinematics, only the radial component remains,

$$
\ell_r
=
u_\theta
\frac1r\frac{\partial(ru_\theta)}{\partial r}
+
u_z\frac{\partial u_z}{\partial r}.
$$

The isotropic baseline is

$$
\ell_r^{(\mathrm{iso})}
=
u_z^{(\mathrm{iso})}
\frac{\partial u_z^{(\mathrm{iso})}}{\partial r},
$$

and the mechanism-isolating quantity is

$$
\Delta\ell_r
=
\ell_r^{(\mathrm{aniso})}
-
\ell_r^{(\mathrm{iso})}.
$$

Because $\ell_r$ is quadratic, the physical real fields must be reconstructed before multiplication.

In [ ]:
# Reconstruct one harmonic over one period and compare anisotropic and isotropic Lamb fields.
phase = np.linspace(0.0, 1.0, 360, endpoint=False)
exp1 = np.exp(2j*np.pi*phase)

# Interpolate isotropic solution onto the Case A radial grid.
Uz_iso_interp = np.interp(x, x_iso, Uz_iso_num.real) + 1j*np.interp(x, x_iso, Uz_iso_num.imag)
Uzp_iso_exact = np.gradient(Uz_iso_interp, x)

# Real reconstructed fields, shape (time, radius).
uz_an = np.real(exp1[:, None] * Uz[None, :])
uzp_an = np.real(exp1[:, None] * Uzp[None, :])
uth_an = np.real(exp1[:, None] * Uth[None, :])
uthp_an = np.real(exp1[:, None] * Uthp[None, :])

omega_z_an = uthp_an + uth_an/x[None, :]
ell_an = uth_an*omega_z_an + uz_an*uzp_an

uz_iso_t = np.real(exp1[:, None] * Uz_iso_interp[None, :])
uzp_iso_t = np.real(exp1[:, None] * Uzp_iso_exact[None, :])
ell_iso = uz_iso_t*uzp_iso_t

delta_ell = ell_an - ell_iso

# Select four deterministic phases.
phase_indices = [0, 90, 180, 270]
styles_phase = [
    (BLACK, "-", r"$t/T=0$"),
    (DARK, "--", r"$t/T=0.25$"),
    (MID, "-.", r"$t/T=0.50$"),
    (LIGHT, ":", r"$t/T=0.75$"),
]

fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.1))

for idx, (gray, ls, label) in zip(phase_indices, styles_phase):
    axes[0].plot(
        x, ell_an[idx],
        color=gray, linestyle=ls, label=label
    )
    axes[1].plot(
        x, delta_ell[idx],
        color=gray, linestyle=ls, label=label
    )

axes[0].set_xlabel(r"Normalized radius, $x=r/R$")
axes[0].set_ylabel(r"Dimensionless $\ell_r^{(\mathrm{aniso})}$")
axes[0].legend(frameon=False)
clean_axes(axes[0])

axes[1].set_xlabel(r"Normalized radius, $x=r/R$")
axes[1].set_ylabel(r"Dimensionless $\Delta\ell_r$")
axes[1].legend(frameon=False)
clean_axes(axes[1])

fig.tight_layout(w_pad=1.3)
save_figure(fig, "ch04_lamb_increment")
plt.show()

The left panel contains the complete radial Lamb-vector component for the anisotropic Case A solution. The right panel subtracts the isotropic Womersley baseline.

The subtraction is essential. The classical axial shear already produces a finite radial Lamb-vector term. The constitutive claim concerns the **increment** that appears when the azimuthal channel is opened.

# Nonlinear force spectra and near-wall localization

## Why the force spectrum is nonlinear even when the velocity problem is linear

The harmonic boundary-value problem is linear for prescribed coefficients. The derived force-density signal is not, because

$$
\boldsymbol\ell(t)
=
\mathbf u(t)\times\boldsymbol\omega(t).
$$

When several harmonics are reconstructed, the product contains sum and difference frequencies.

The notebook therefore follows the chapter and Chapter 8 exactly:

1. solve each harmonic independently;
2. reconstruct the real velocity and vorticity fields;
3. form $\ell_r(r,t)$ in physical time;
4. only then take an FFT of the completed nonlinear signal.

## Near-wall localization and harmonic Womersley scaling

For harmonic $m$,

$$
\delta_{W,m}
=
\sqrt{\frac{2\nu_{zz}}{m\Omega}}
=
\frac{\delta_{W,1}}{\sqrt m}.
$$

This identifies a localization scale. It does not determine the amplitude of the harmonic response, which still depends on forcing, constitutive coupling, phase, and the solution of the coupled system.

In [ ]:
# Demonstrate harmonic near-wall localization with the canonical constitutive coupling.
m_values = [1, 2, 4, 8]
localization_rows = []

fig, ax = plt.subplots(figsize=(5.9, 3.35))

styles_m = [
    (BLACK, "-", r"$m=1$"),
    (DARK, "--", r"$m=2$"),
    (MID, "-.", r"$m=4$"),
    (LIGHT, ":", r"$m=8$"),
]

for mm, (gray, ls, label) in zip(m_values, styles_m):
    sol_m = solve_ch4_harmonic(
        alpha=CASE_A_ALPHA,
        m=mm,
        a_m=1.0,
        epsilon=1e-5,
        tol=5e-7,
    )
    xx = np.linspace(1e-5, 1.0, 900)
    Uz_m, Uzp_m, Uth_m, Uthp_m = evaluate_solution(sol_m, xx)
    omega_z_m = Uthp_m + Uth_m/xx

    profile = np.abs(omega_z_m)
    profile /= np.max(profile)

    ax.plot(
        1.0-xx, profile,
        color=gray, linestyle=ls, label=label
    )

    localization_rows.append({
        "m": mm,
        "alpha_m": np.sqrt(mm)*CASE_A_ALPHA,
        "deltaW_m_over_R": np.sqrt(2.0)/(np.sqrt(mm)*CASE_A_ALPHA),
        "max_abs_Utheta": float(np.max(np.abs(Uth_m))),
        "max_abs_omega_z_scaled": float(np.max(np.abs(omega_z_m))),
    })

ax.set_xlim(0.0, 0.35)
ax.set_xlabel(r"Distance from wall, $(R-r)/R$")
ax.set_ylabel(r"$|\widehat\omega_{z,m}|/\max|\widehat\omega_{z,m}|$")
ax.legend(frameon=False)
clean_axes(ax)
fig.tight_layout()

save_figure(fig, "ch04_harmonic_near_wall_localization")
plt.show()

localization_df = pd.DataFrame(localization_rows)
localization_df.to_csv(DATA_DIR / "ch04_harmonic_localization.csv", index=False)
display(localization_df)

# From force density to an endothelial-scale control volume

Define

$$
\delta_{\mathrm{EC}}
=
\frac{V_{\mathrm{EC}}}{A_{\mathrm{EC}}}.
$$

The chapter distinguishes

$$
F_{r,\mathrm{EC}}^{\mathrm{mag}}(t)
=
A_{\mathrm{EC}}
\int_{R-\delta_{\mathrm{EC}}}^{R}
|f_{L,r}(r,t)|\,dr,
$$

from

$$
F_{r,\mathrm{EC}}^{\mathrm{signed}}(t)
=
A_{\mathrm{EC}}
\int_{R-\delta_{\mathrm{EC}}}^{R}
f_{L,r}(r,t)\,dr.
$$

The first accumulates local magnitude; the second retains directional cancellation.

No endothelial dimensions are invented in this notebook. The computation uses the dimensionless thickness $\delta_{\mathrm{EC}}/R$ and reports normalized quantities unless VascuQuest supplies all dimensional scales required for a particular reconstruction.

In [ ]:
# Canonical signed-versus-magnitude pillbox descriptors.
delta_EC_over_R = 0.05
mask = x >= (1.0 - delta_EC_over_R)
x_wall = x[mask]

# rho, A_EC and U0^2 factors are omitted here, so the plotted quantities are
# normalized descriptors of the Chapter 4 integral operation.
F_mag = np.trapz(np.abs(ell_an[:, mask]), x_wall, axis=1)
F_signed = np.trapz(ell_an[:, mask], x_wall, axis=1)
F_mag_iso = np.trapz(np.abs(ell_iso[:, mask]), x_wall, axis=1)

fig, ax = plt.subplots(figsize=(6.0, 3.3))

ax.plot(phase, F_mag, color=BLACK, linestyle="-",
        label=r"anisotropic $F_{r,\mathrm{EC}}^{\mathrm{mag}}$")
ax.plot(phase, F_signed, color=DARK, linestyle="--",
        label=r"anisotropic $F_{r,\mathrm{EC}}^{\mathrm{signed}}$")
ax.plot(phase, F_mag_iso, color=LIGHT, linestyle=":",
        label=r"isotropic magnitude baseline")

ax.set_xlabel(r"Normalized time, $t/T$")
ax.set_ylabel("Normalized near-wall integral")
ax.legend(frameon=False)
clean_axes(ax)
fig.tight_layout()

save_figure(fig, "ch04_endothelial_control_volume")
plt.show()

The magnitude and signed measures differ because they answer different questions. Their difference must not be hidden by calling both of them a “force” without qualification.

They remain fluid control-volume descriptors. Exact endothelial surface loading remains a traction problem.

# VascuQuest exploration: the same constitutive mechanism across physiological $\alpha$

The book's Case A coupling is now held fixed while VascuQuest supplies the subject/site values of $R$ and heart rate.

This is the correct direction of inference:

- **VascuQuest supplies the classical pulsatile scale $\alpha$;**
- **the book supplies the controlled constitutive ratios;**
- **the notebook computes the resulting normalized transverse response.**

The notebook does **not** infer $\mathcal A_{ij}$ from age, arterial site, or PWDB waveforms.

In [ ]:
# Build the physiological alpha population from PWDB.
meta = subject_meta.set_index("subject_id")
population_rows = []

for site in SITES:
    print("Processing", SITE_LABELS[site])
    ids_a, A_matrix = load_waveform_matrix(site, "A")

    for sid, A_row in zip(ids_a, A_matrix):
        if sid not in meta.index:
            continue

        valid = np.isfinite(A_row)
        if valid.sum() < 16:
            continue

        mean_area = float(np.mean(A_row[valid]))
        R_value = np.sqrt(mean_area/np.pi)
        heart_rate_bpm = float(meta.loc[sid, "heart_rate_bpm"])
        age_years = float(meta.loc[sid, "age_years"])

        T_value = 60.0/heart_rate_bpm
        Omega_value = 2.0*np.pi/T_value
        alpha_value = R_value*np.sqrt(Omega_value/nu_zz)

        population_rows.append({
            "subject_id": sid,
            "age_years": age_years,
            "site": site,
            "R_m": R_value,
            "heart_rate_bpm": heart_rate_bpm,
            "alpha": alpha_value,
        })

population_df = pd.DataFrame(population_rows)
population_df.to_csv(DATA_DIR / "ch04_population_alpha.csv", index=False)

display(
    population_df.groupby("site")["alpha"]
    .agg(["count", "median", "min", "max"])
    .reindex(SITES)
    .rename(index=SITE_LABELS)
)

In [ ]:
# Precompute the canonical transverse-response function over the alpha range.
alpha_min = max(0.5, 0.85*population_df["alpha"].min())
alpha_max = 1.15*population_df["alpha"].max()
alpha_grid = np.geomspace(alpha_min, alpha_max, 70)

response_rows = []
for k, a in enumerate(alpha_grid):
    if k % 10 == 0:
        print(f"alpha grid {k+1}/{len(alpha_grid)}")
    sol_a = solve_ch4_harmonic(
        alpha=float(a),
        m=1,
        a_m=1.0,
        epsilon=1e-5,
        tol=1e-6,
        initial_nodes=220,
    )
    xx = np.linspace(1e-5, 1.0, 500)
    Uz_a, _, Uth_a, _ = evaluate_solution(sol_a, xx)

    response_rows.append({
        "alpha": float(a),
        "max_abs_Uz": float(np.max(np.abs(Uz_a))),
        "max_abs_Utheta": float(np.max(np.abs(Uth_a))),
        "transverse_to_axial_ratio":
            float(np.max(np.abs(Uth_a))/np.max(np.abs(Uz_a))),
    })

response_grid_df = pd.DataFrame(response_rows)
response_grid_df.to_csv(DATA_DIR / "ch04_canonical_response_grid.csv", index=False)

interp_ratio = PchipInterpolator(
    response_grid_df["alpha"],
    response_grid_df["transverse_to_axial_ratio"],
    extrapolate=False,
)

population_df["transverse_to_axial_ratio"] = interp_ratio(
    population_df["alpha"].to_numpy()
)

fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.15))

axes[0].plot(
    response_grid_df["alpha"],
    response_grid_df["transverse_to_axial_ratio"],
    color=BLACK
)
axes[0].scatter(
    population_df["alpha"],
    population_df["transverse_to_axial_ratio"],
    s=7, facecolors="none", edgecolors=LIGHT, linewidths=0.45
)
axes[0].set_xscale("log")
axes[0].set_xlabel(r"Womersley number, $\alpha$")
axes[0].set_ylabel(r"$\max|\widehat U_\theta|/\max|\widehat U_z|$")
clean_axes(axes[0])

site_groups = [
    population_df.loc[
        population_df["site"] == site,
        "transverse_to_axial_ratio"
    ].dropna().to_numpy()
    for site in SITES
]
axes[1].boxplot(
    site_groups,
    labels=[SITE_LABELS[s] for s in SITES],
    showfliers=False,
    whis=(5, 95),
    widths=0.58,
    medianprops={"color": BLACK, "linewidth": 1.3},
    boxprops={"color": BLACK, "linewidth": 0.9},
    whiskerprops={"color": DARK, "linewidth": 0.8},
    capprops={"color": DARK, "linewidth": 0.8},
)
axes[1].tick_params(axis="x", rotation=52)
axes[1].set_ylabel(r"Canonical $\max|\widehat U_\theta|/\max|\widehat U_z|$")
clean_axes(axes[1])

fig.tight_layout(w_pad=1.3)
save_figure(fig, "ch04_vascuquest_population_transverse_response")
plt.show()

The left panel is a **computed model-response curve** with VascuQuest subject/site values overlaid. The right panel groups the same computed response by arterial site.

The population does not provide anisotropy coefficients. The variation arises because the fixed canonical constitutive mechanism is evaluated at different VascuQuest-derived values of $\alpha$.

This is therefore a parameter-context result, not a physiological calibration of anisotropy.

In [ ]:
# Age-group comparison using the same fixed canonical constitutive coupling.
age_summary = (
    population_df
    .groupby(["age_years", "site"])
    ["transverse_to_axial_ratio"]
    .median()
    .reset_index()
)

AGE_SITES = ["AorticRoot", "Carotid", "Femoral", "Radial"]
styles_age = [
    (BLACK, "-", "o"),
    (DARK, "--", "s"),
    (MID, "-.", "^"),
    (LIGHT, ":", "D"),
]

fig, ax = plt.subplots(figsize=(6.0, 3.35))

for site, (gray, ls, marker) in zip(AGE_SITES, styles_age):
    sub = age_summary.loc[age_summary["site"] == site]
    ax.plot(
        sub["age_years"],
        sub["transverse_to_axial_ratio"],
        color=gray, linestyle=ls, marker=marker, markersize=4,
        label=SITE_LABELS[site],
    )

ax.set_xlabel("PWDB source age (years)")
ax.set_ylabel(r"Median canonical $\max|\widehat U_\theta|/\max|\widehat U_z|$")
ax.legend(frameon=False)
clean_axes(ax)
fig.tight_layout()

save_figure(fig, "ch04_age_group_transverse_response")
plt.show()

The age-group plot is descriptive. Age changes the PWDB distributions of radius and heart rate, and those quantities change $\alpha$. The notebook does not infer age-dependent constitutive anisotropy.

Any apparent age trend therefore means only:

> under the same canonical constitutive ratios, the Chapter 4 transverse response changes as the classical pulsatile scale changes across the PWDB age strata.

# Representative multiharmonic VascuQuest reconstruction

To connect the harmonic Chapter 4 solver to a physiological waveform, use the representative aortic-root $Q(t)$ signal.

For each harmonic:

1. solve the coupled Chapter 4 problem with unit dimensionless pressure forcing $a_m=1$;
2. integrate the unit axial response over the cross-section;
3. infer the physical $\widehat G_m$ required to reproduce the observed $\widehat Q_m$;
4. scale both $\widehat u_{z,m}$ and $\widehat u_{\theta,m}$ by that same pressure-gradient amplitude;
5. reconstruct the real velocity and vorticity fields;
6. form $\ell_r$ and its spectrum only after reconstruction.

The constitutive ratios remain the canonical Case A values. This is a model projection, not a calibration.

In [ ]:
# Representative aortic-root Q(t).
REP_SITE = "AorticRoot"

ids_u, U_matrix = load_waveform_matrix(REP_SITE, "U")
ids_a, A_matrix = load_waveform_matrix(REP_SITE, "A")
assert np.array_equal(ids_u, ids_a)

idx = np.where(ids_u == representative_subject)[0]
if len(idx) != 1:
    raise RuntimeError("Representative subject not found uniquely.")
idx = int(idx[0])

valid = np.isfinite(U_matrix[idx]) & np.isfinite(A_matrix[idx])
U_values = U_matrix[idx][valid]
A_values = A_matrix[idx][valid]
Q_values = U_values * A_values

R_rep = float(np.sqrt(np.mean(A_values)/np.pi))
hr_rep = float(meta.loc[representative_subject, "heart_rate_bpm"])
T_rep = 60.0/hr_rep
Omega_rep = 2.0*np.pi/T_rep
alpha_rep = R_rep*np.sqrt(Omega_rep/nu_zz)

# Fourier representation in the book convention:
# real signal = Q0 + Re(sum Qhat_m exp(i m Omega t)).
Q_coeff = np.fft.rfft(Q_values) / len(Q_values)
M = min(8, len(Q_coeff)-1)
Q0 = float(np.real(Q_coeff[0]))
Qhat = 2.0*Q_coeff[1:M+1]
m_vec = np.arange(1, M+1)

print("Representative subject:", representative_subject)
print(f"R = {R_rep*1e3:.3f} mm")
print(f"heart rate = {hr_rep:.1f} min^-1")
print(f"alpha = {alpha_rep:.3f}")
print("retained positive harmonics:", M)

In [ ]:
# Solve each retained harmonic with a_m=1, then scale the physical solution
# so its cross-sectional flow harmonic equals the VascuQuest Qhat_m.
x_multi = np.linspace(1e-4, 1.0, 520)

Uz_hat_phys = np.zeros((M, len(x_multi)), dtype=complex)
Uth_hat_phys = np.zeros((M, len(x_multi)), dtype=complex)
Uzp_hat_phys = np.zeros((M, len(x_multi)), dtype=complex)
Uthp_hat_phys = np.zeros((M, len(x_multi)), dtype=complex)
Ghat_phys = np.zeros(M, dtype=complex)

for k, (mm, qh) in enumerate(zip(m_vec, Qhat)):
    sol_m = solve_ch4_harmonic(
        alpha=alpha_rep,
        m=int(mm),
        a_m=1.0,
        epsilon=1e-4,
        tol=7e-7,
        initial_nodes=260,
    )
    Uz_u, Uzp_u, Uth_u, Uthp_u = evaluate_solution(sol_m, x_multi)

    # With unit a_m, physical velocity for pressure gradient Ghat is
    # (R^2 Ghat / (rho nu_zz)) * U_unit.
    flow_shape = 2.0*np.pi*np.trapz(Uz_u*x_multi, x_multi)
    Q_per_G = (
        R_rep**4/(rho*nu_zz)
    ) * flow_shape

    Ghat = qh / Q_per_G
    velocity_scale = R_rep**2 * Ghat / (rho*nu_zz)

    Ghat_phys[k] = Ghat
    Uz_hat_phys[k] = velocity_scale * Uz_u
    Uth_hat_phys[k] = velocity_scale * Uth_u
    Uzp_hat_phys[k] = velocity_scale * Uzp_u / R_rep
    Uthp_hat_phys[k] = velocity_scale * Uthp_u / R_rep

phase_multi = np.linspace(0.0, 1.0, len(Q_values), endpoint=False)
E = np.exp(2j*np.pi*np.outer(phase_multi, m_vec))

# The mean is retained only in the observed Q(t) comparison.
# The Chapter 4 nonlinear increment below is formed from the oscillatory
# harmonics because the chapter's constitutive benchmark is harmonic.
Q_recon_harm = np.real(E @ Qhat)
Q_recon = Q0 + Q_recon_harm

# Real oscillatory velocity fields.
uz = np.real(E @ Uz_hat_phys)
uth = np.real(E @ Uth_hat_phys)
uzp = np.real(E @ Uzp_hat_phys)
uthp = np.real(E @ Uthp_hat_phys)

omega_theta = -uzp
omega_z = uthp + uth/(R_rep*x_multi[None, :])

ell_an = uth*omega_z + uz*uzp

# Isotropic baseline driven by the same Q harmonics:
# solve with coupling off and repeat the flow matching.
Uz_iso_hat_phys = np.zeros_like(Uz_hat_phys)
Uzp_iso_hat_phys = np.zeros_like(Uzp_hat_phys)

for k, (mm, qh) in enumerate(zip(m_vec, Qhat)):
    sol_m = solve_ch4_harmonic(
        alpha=alpha_rep,
        m=int(mm),
        a_m=1.0,
        Azt=0.0, Atz=0.0, Att=1.0,
        epsilon=1e-4,
        tol=7e-7,
        initial_nodes=240,
    )
    Uz_u, Uzp_u, _, _ = evaluate_solution(sol_m, x_multi)

    flow_shape = 2.0*np.pi*np.trapz(Uz_u*x_multi, x_multi)
    Q_per_G = R_rep**4/(rho*nu_zz) * flow_shape
    Ghat_iso = qh / Q_per_G
    velocity_scale = R_rep**2 * Ghat_iso / (rho*nu_zz)

    Uz_iso_hat_phys[k] = velocity_scale * Uz_u
    Uzp_iso_hat_phys[k] = velocity_scale * Uzp_u / R_rep

uz_iso = np.real(E @ Uz_iso_hat_phys)
uzp_iso = np.real(E @ Uzp_iso_hat_phys)
ell_iso = uz_iso*uzp_iso

delta_ell = ell_an - ell_iso

recon_df = pd.DataFrame({
    "phase": phase_multi,
    "Q_source_m3_s": Q_values,
    "Q_reconstructed_m3_s": Q_recon,
})
recon_df.to_csv(DATA_DIR / "ch04_representative_Q_reconstruction.csv", index=False)

fig, axes = plt.subplots(3, 1, figsize=(6.3, 5.7), sharex=True)

axes[0].plot(
    phase_multi, Q_values*1e6,
    color=LIGHT, linestyle=":", label="source"
)
axes[0].plot(
    phase_multi, Q_recon*1e6,
    color=BLACK, label=f"{M}-harmonic reconstruction"
)
axes[0].set_ylabel(r"$Q$ (mL s$^{-1}$)")
axes[0].legend(frameon=False)
clean_axes(axes[0])

# Report velocities at a near-wall location x=0.9.
j90 = int(np.argmin(np.abs(x_multi-0.90)))
axes[1].plot(
    phase_multi, uz[:, j90],
    color=BLACK, label=r"$u_z$ at $x=0.9$"
)
axes[1].plot(
    phase_multi, uth[:, j90],
    color=DARK, linestyle="--", label=r"$u_\theta$ at $x=0.9$"
)
axes[1].set_ylabel(r"Velocity (m s$^{-1}$)")
axes[1].legend(frameon=False)
clean_axes(axes[1])

axes[2].plot(
    phase_multi, delta_ell[:, j90],
    color=BLACK
)
axes[2].set_ylabel(r"$\Delta\ell_r$ (m s$^{-2}$)")
axes[2].set_xlabel(r"Normalized time, $t/T$")
clean_axes(axes[2])

fig.tight_layout(h_pad=0.45)
save_figure(fig, "ch04_representative_multiharmonic_projection")
plt.show()

In [ ]:
# Nonlinear spectrum of the anisotropic increment after real-field reconstruction.
# Use the near-wall x=0.9 signal where gradients are dynamically visible.
signal = delta_ell[:, j90]
coeff = np.fft.rfft(signal) / len(signal)
harmonic_number = np.arange(len(coeff))
amplitude = 2.0*np.abs(coeff)
amplitude[0] = np.abs(coeff[0])

max_show = min(2*M + 2, len(amplitude))

fig, ax = plt.subplots(figsize=(5.8, 3.25))
ax.stem(
    harmonic_number[:max_show],
    amplitude[:max_show],
    linefmt="k-", markerfmt="ko", basefmt=" "
)
ax.axvline(M, color=LIGHT, linestyle=":", linewidth=1.0)
ax.set_xlabel(r"Harmonic number, $m$")
ax.set_ylabel(r"Spectrum of $\Delta\ell_r$ at $x=0.9$")
ax.set_xticks(harmonic_number[:max_show])
clean_axes(ax)
fig.tight_layout()

save_figure(fig, "ch04_nonlinear_force_spectrum")
plt.show()

The vertical reference marks the highest harmonic retained in the **linear velocity reconstruction**. Content above that index can appear in $\Delta\ell_r$ because the Lamb-vector operation is quadratic.

This is not a linear instability and not Chapter 5 spectral evolution. It is the deterministic sum- and difference-frequency content of a nonlinear observable formed from linearly solved harmonic fields.

In [ ]:
# Dimensional endothelial-scale normalized-thickness comparison for the
# representative multiharmonic projection.
delta_ratios = [0.01, 0.025, 0.05, 0.10]
pillbox_rows = []

fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.15))

for d_ratio, gray, ls in zip(
    delta_ratios,
    [BLACK, DARK, MID, LIGHT],
    ["-", "--", "-.", ":"],
):
    mask = x_multi >= 1.0-d_ratio
    r_local = R_rep*x_multi[mask]

    # A_EC is factored out because no endothelial footprint is imposed.
    Fmag_per_area = np.trapz(
        rho*np.abs(ell_an[:, mask]), r_local, axis=1
    )
    Fsigned_per_area = np.trapz(
        rho*ell_an[:, mask], r_local, axis=1
    )

    axes[0].plot(
        phase_multi, Fmag_per_area,
        color=gray, linestyle=ls,
        label=rf"$\delta_{{\mathrm{{EC}}}}/R={d_ratio:g}$"
    )
    axes[1].plot(
        phase_multi, Fsigned_per_area,
        color=gray, linestyle=ls,
        label=rf"$\delta_{{\mathrm{{EC}}}}/R={d_ratio:g}$"
    )

    pillbox_rows.append({
        "delta_EC_over_R": d_ratio,
        "max_Fmag_per_area_Pa": float(np.max(Fmag_per_area)),
        "max_abs_Fsigned_per_area_Pa":
            float(np.max(np.abs(Fsigned_per_area))),
    })

axes[0].set_xlabel(r"Normalized time, $t/T$")
axes[0].set_ylabel(r"$F_{r,\mathrm{EC}}^{\mathrm{mag}}/A_{\mathrm{EC}}$ (Pa)")
axes[0].legend(frameon=False)
clean_axes(axes[0])

axes[1].set_xlabel(r"Normalized time, $t/T$")
axes[1].set_ylabel(r"$F_{r,\mathrm{EC}}^{\mathrm{signed}}/A_{\mathrm{EC}}$ (Pa)")
axes[1].legend(frameon=False)
clean_axes(axes[1])

fig.tight_layout(w_pad=1.25)
save_figure(fig, "ch04_representative_endothelial_pillbox")
plt.show()

pillbox_df = pd.DataFrame(pillbox_rows)
pillbox_df.to_csv(DATA_DIR / "ch04_pillbox_sensitivity.csv", index=False)
display(pillbox_df)

Dividing by $A_{\mathrm{EC}}$ leaves the force-per-footprint-area measure with units of pressure and avoids inventing an endothelial footprint.

The dependence on $\delta_{\mathrm{EC}}/R$ is shown explicitly because the spatial support of the near-wall control volume is part of the definition of the descriptor.

# What the constitutive extension establishes

The notebook reproduces the controlled Chapter 4 conclusion:

- pressure forcing acts directly only in the axial equation;
- off-diagonal shear coupling transmits that forcing into $u_\theta$;
- $u_\theta$ opens the axial-vorticity component $\omega_z$;
- the resulting velocity–vorticity field modifies $\ell_r$;
- the isotropic counterfactual removes the transverse mode and recovers the scalar Womersley state;
- nonlinear force-density spectra arise when reconstructed velocity and vorticity fields are multiplied;
- higher harmonic number narrows the near-wall oscillatory scale through
  $$
  \delta_{W,m}
  =
  \delta_{W,1}/\sqrt m.
  $$

The conclusion remains deliberately narrow. The notebook does not calibrate blood anisotropy, does not attribute the mechanism to red-cell microstructure, and does not introduce curvature, wall compliance, or Chapter 5 geometry-dependent spectral dynamics.

# What the reader should learn

1. **The transverse degree of freedom is opened constitutively, not geometrically.** Geometry is held at the straight rigid reference state.

2. **The four radial operators are not interchangeable.** The two cross-divergence operators follow from the cylindrical stress divergences of the stated constitutive law.

3. **The isotropic limit is a required mechanism-off test.** When the off-diagonal ratios vanish, $u_\theta$ and $\omega_z$ vanish and the scalar Womersley solution is recovered.

4. **Classical Womersley flow already contains $\omega_\theta$.** The constitutive extension specifically opens the additional axial-vorticity channel $\omega_z$.

5. **The relevant inertial comparison is the anisotropic increment**
   $$
   \Delta\ell_r
   =
   \ell_r^{(\mathrm{aniso})}
   -
   \ell_r^{(\mathrm{iso})},
   $$
   not the raw Lamb-vector field alone.

6. **Linear harmonic velocity solutions can generate a richer nonlinear force spectrum.** Reconstruction must precede multiplication.

7. **Near-wall localization strengthens with harmonic number, but localization is not amplitude.** The response amplitude still depends on forcing and constitutive coupling.

8. **VascuQuest supplies physiological classical inputs, not anisotropy coefficients.** The population analysis shows how one fixed canonical mechanism behaves across VascuQuest-derived $\alpha$ values.

# Chapter-enrichment candidates

The notebook produces twelve principal figures.

**Candidate 1 — Case A velocity components.**  
Already represented in Chapter 4. Use only as a possible replacement for the existing figure.

**Candidate 2 — isotropic recovery.**  
Strong verification figure but probably notebook-only because the book already states the counterfactual analytically and Chapter 8 reports its numerical verification.

**Candidate 3 — constitutive-coupling sensitivity.**  
Potentially strong book candidate. It shows continuously how the transverse response opens from zero while remaining inside the admissible constitutive family.

**Candidate 4 — vorticity channels.**  
Strong book candidate if a visual is needed to distinguish the pre-existing $\omega_\theta$ from the newly opened $\omega_z$.

**Candidate 5 — Lamb-vector baseline and anisotropic increment.**  
Strong book candidate because it visualizes exactly what $\Delta\ell_r$ isolates.

**Candidate 6 — harmonic near-wall localization.**  
Potentially strong book candidate. It connects the Chapter 4 force-density discussion directly to $\delta_{W,m}=\delta_{W,1}/\sqrt m$.

**Candidate 7 — canonical endothelial control-volume comparison.**  
Useful but probably notebook-only unless the distinction between signed and magnitude accumulation needs a figure in the chapter.

**Candidate 8 — VascuQuest population transverse-response map.**  
Potential book candidate if it adds useful physiological parameter context. Its caption must state that the anisotropy ratios are fixed canonical verification values.

**Candidate 9 — age-group transverse-response comparison.**  
Primarily notebook material. It is descriptive of the PWDB-derived $\alpha$ variation under one fixed canonical constitutive coupling and must not be presented as age-dependent anisotropy.

**Candidate 10 — representative multiharmonic projection.**  
Strong notebook result; possible book candidate if the chapter benefits from a concrete time-domain view of $u_\theta$ and $\Delta\ell_r$ under physiological waveform forcing.

**Candidate 11 — nonlinear spectrum of $\Delta\ell_r$.**  
Potentially strong book candidate because it directly demonstrates sum- and difference-frequency content generated by the quadratic observable after linear harmonic reconstruction.

**Candidate 12 — representative endothelial-pillbox sensitivity.**  
Primarily notebook material unless the chapter needs quantitative scale dependence of the near-wall control-volume descriptor.

No figure is promoted automatically. Existing figures should be replaced rather than duplicated when the notebook version serves the same purpose better.

In [ ]:
# Reproducibility record.
manifest = {
    "book": "Nonlinear Arterial Hemodynamics",
    "chapter": 4,
    "chapter_title": "Constitutive Anisotropy and Transverse Dynamics",
    "vascuquest_git_ref": VQ_GIT_REF,
    "pwdb_record_id": PWDB_RECORD_ID,
    "pwdb_doi": PWDB_DOI,
    "rho_kg_m3": rho,
    "mu_Pa_s": mu,
    "nu_zz_m2_s": nu_zz,
    "canonical_A_ztheta": A_ztheta,
    "canonical_A_thetaz": A_thetaz,
    "canonical_A_thetatheta": A_thetatheta,
    "case_A_alpha": CASE_A_ALPHA,
    "case_A_m": CASE_A_M,
    "representative_subject_id": representative_subject,
    "representative_age_years": float(target_age),
    "representative_site": REP_SITE,
    "representative_R_m": R_rep,
    "representative_heart_rate_bpm": hr_rep,
    "representative_alpha": alpha_rep,
    "retained_positive_harmonics": int(M),
    "sites": SITES,
    "source_age_strata_years": [float(x) for x in source_ages],
    "python": sys.version.split()[0],
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "matplotlib": matplotlib.__version__,
    "execution_status": "completed to this cell",
    "qualification": (
        "PWDB supplies physiological radius, heart rate, age, and Q(t). "
        "The anisotropy ratios are fixed canonical book verification parameters, "
        "not PWDB variables or calibrated arterial population distributions. "
        "All transverse velocities, axial vorticity, Lamb-vector increments, "
        "and endothelial-scale descriptors are Chapter 4 model projections."
    ),
}
(META_DIR / "reproducibility_manifest.json").write_text(
    json.dumps(manifest, indent=2), encoding="utf-8"
)

print(json.dumps(manifest, indent=2))
print("\nGenerated PDF figures:")
for path in sorted(FIG_DIR.glob("*.pdf")):
    print(" -", path.name)

# Reproducibility record

A successful **Run all** execution writes:

- B&W vector PDF figures and high-resolution PNG previews;
- the Case A centreline-regularity/benchmark check;
- the constitutive-coupling sensitivity sweep;
- harmonic localization diagnostics;
- VascuQuest-derived $\alpha$ population data;
- the canonical response grid used for population projection;
- representative multiharmonic reconstruction data;
- endothelial-pillbox sensitivity data;
- VascuQuest/PWDB verification metadata;
- a final reproducibility manifest.

No interactive branch, hidden parameter tuning, or manually selected subject is used.